# PS LiDAR - Laboratorio de Desarrollo

Este notebook sirve para probar los **ladrillos** del proyecto de forma interactiva.

**Ladrillos disponibles:**
- Brick 1: Carga de Datos (`PointCloudLoader`)
- Brick 2: Detección de Normalización (`detect_normalization`)
- Brick 3: Filtrado de Suelo (`classify_ground`)

In [ ]:
import os
import sys
import time
from pathlib import Path

# Asegurar que podemos importar desde la carpeta src
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Importar todos los componentes disponibles
from src.core import (
    PointCloudLoader,
    detect_normalization,
    NormalizationStatus,
    classify_ground,
    get_ground_mask,
    get_ground_points,
)

print("✓ Módulos importados correctamente")

---
## 1. Cargar Archivo (Brick 1)
Selecciona un archivo `.las` o `.laz` para empezar.

In [ ]:
# Configura aquí la ruta a tu archivo
FILE_PATH = "../external_references/artemis_treeiso/data/LPine1_demo.laz"

# Cargar
loader = PointCloudLoader(FILE_PATH)
loader.load()

# Mostrar metadatos
meta = loader.get_metadata()
print(f"Archivo: {meta['filename']}")
print(f"Puntos: {meta['point_count']:,}")
print(f"Tamaño: {meta['file_size_mb']} MB")
print(f"Rango Z: {meta['min_coords'][2]:.2f}m a {meta['max_coords'][2]:.2f}m")

In [ ]:
# Obtener coordenadas XYZ
xyz = loader.get_xyz()
print(f"Shape: {xyz.shape}")
print(f"Dtype: {xyz.dtype}")
print(f"Memoria: {xyz.nbytes / (1024**2):.2f} MB")

---
## 2. Análisis de Normalización (Brick 2)
Verificar si la nube ya tiene las alturas relativas al suelo.

In [ ]:
# Analizar normalización
analysis = detect_normalization(xyz)

print(f"Estatus: {analysis.status.value.upper()}")
print(f"Confianza: {analysis.confidence:.1%}")
print(f"¿Normalizada?: {analysis.is_normalized}")
print(f"")
print(f"Estadísticas Z:")
print(f"  Min: {analysis.z_min:.2f}m")
print(f"  Max: {analysis.z_max:.2f}m")
print(f"  Rango: {analysis.z_range:.2f}m")
print(f"  5th percentil: {analysis.percentile_5:.2f}m")
print(f"")
print(f"Razonamiento:")
for reason in analysis.reasoning.split("; "):
    print(f"  • {reason}")

---
## 3. Filtrado de Suelo - CSF (Brick 3)
Separar puntos de suelo de la vegetación usando Cloth Simulation Filter.

In [ ]:
# Ejecutar filtrado de suelo
print("Ejecutando Cloth Simulation Filter...")
t0 = time.perf_counter()

result = classify_ground(
    xyz,
    cloth_resolution=1.0,  # Resolución del cloth en metros
    rigidness=1,           # 1=plano, 2=relieve, 3=escarpado
    class_threshold=0.5,   # Distancia umbral para clasificar como suelo
    slope_smooth=True,     # Suavizado para pendientes
)

elapsed = time.perf_counter() - t0
print(f"✓ Completado en {elapsed:.2f}s")
print(f"")
print(f"Resultados:")
print(f"  Suelo: {result.n_ground:,} puntos ({result.ground_ratio:.1%})")
print(f"  Vegetación: {result.n_off_ground:,} puntos ({1 - result.ground_ratio:.1%})")

In [ ]:
# Analizar la distribución Z de los puntos de suelo
ground_z = xyz[result.ground_indices, 2]
veg_z = xyz[result.off_ground_indices, 2]

print("Distribución Z del suelo:")
print(f"  Min: {ground_z.min():.2f}m")
print(f"  Max: {ground_z.max():.2f}m")
print(f"  Media: {ground_z.mean():.2f}m")
print(f"")
print("Distribución Z de vegetación:")
print(f"  Min: {veg_z.min():.2f}m")
print(f"  Max: {veg_z.max():.2f}m")
print(f"  Media: {veg_z.mean():.2f}m")

In [ ]:
# Alternativa: Obtener directamente los puntos de suelo
ground_points = get_ground_points(xyz, cloth_resolution=1.0)
print(f"Puntos de suelo extraídos: {len(ground_points):,}")
print(f"Shape: {ground_points.shape}")

---
## 4. Próximos Pasos

**Brick 4 (pendiente):** Normalización de Altura
- Usar los puntos de suelo para crear un DTM
- Interpolar altura del terreno para cada punto
- Calcular Z_normalizado = Z - DTM

**Brick 5 (pendiente):** Segmentación de Árboles
- Identificar árboles individuales en la nube normalizada